# HAYATE — Colab FastH3
無料T4: 診断とテスト。有料GPU: 条件を満たす場合に4ステップ動画生成を検証します。
このPCでは実行せず、Colabのホスト型ランタイムを使用してください。
全セル実行では大容量モデルを取得しません。生成セルは明示的なフラグで有効にします。

In [ ]:
# 1. ノートブック同梱コードをColab VMへ展開（非公開GitHubの認証不要）
import google.colab
import sys, subprocess, time, json, gzip, base64, hashlib
from pathlib import Path
if sys.platform != 'linux' or not Path('/content').is_dir():
    raise RuntimeError('Google Colabのホスト型ランタイムを選択してください。')
if 'SESSION_STARTED' not in globals():
    SESSION_STARTED = time.monotonic()
REPO = Path('/content/HAYATE-notebook')
PAYLOAD = 'H4sIAAAAAAACCq19jVbbyLLuq+hk1izJGVv4H0PGe12GkIS1CXCBZN9ZhKWRLRm0sSVHkvnZWZx1HuI+4X2S+1VVS2rJMpCzT2YSsNRdXV1dXV2/7R9v7vzQi+Ktm55zl7hbYeT5ib18fLP75pv8F38L96PF7PHLYetTr/X1fM/4f//1f404WqW+8TkIg8/ug/GpZ7hp6odpEIVGeoOX1zfGBzdJvwaeH5mJwT8J1PnSjRPf2MtbWxdxkOLnrR+H/rxhrJIgvAYM38BQrfdu6n6Ifd84is72ACeN3SD0PYL0Vxo5127qO9NosYz9JPnLoI/GvR9c36SJTW3o79kKAy38VhTOH3eNMDLUbIwkWsVT35gFcz8x3Ng30mg1vfE927jA6ESIXepvGJ2GEfuul5TgG7M4Whgu8PXjVrJaLueB7xmJO/MxsySKEwZsWMs48lZTvJo8CjT8mUbhnR+nzgwUuiPKEOkdzGdKqIH6RqvlJWmLxmtIr27DCMJ/+lMM7BqTwE1aMyJLGNpHIIgbWzeB5/mh0fob2oV+TM3TyPAxziOtTWgsgMfcz3GIZkxkrJwXzGagesRN/HnTmEeuB4Tvg/SG25SnHcV46KbGZB5NbwVcr2Es3RS0S3goG23u3dgDQQ1aHOKUvxjI2KgumvXQ+KsJ8iY37rJADh0v/2gan5rGedN4f9U03NAzpu58boBQtpAsYU5yeG6EqkZdAc2vHMF7zMMTlDRa3jIey1sndsGCxlsjdHgyiYDoN4wZhkqMiTu9JVTCKF64c43D729AaYyBdfQCegKyuMHcsJKbKE6NxP/ezLG5jyPws5c+Lv2msQgSZm8ejunapBkZq9C9AwB3Mpf1Zr4F+MUCmxNLsXSDGP12CeZb3liferuAFgYL98HB1i1zkoddQ+zhdHrtdpL6S6fP/4IlRg4xXxylgp4VYEkHbaNmMy3cNA6mamv4i4kP/vLeGdObKEp86f3X78wxrewtM2zyt7+Y0B4RDsRImYpgo4tVPIl4IzdkHidxcB2EoCwJFTDQzS6IusL2Ng6PL0YgPwTFb/ySOuFXNzT8hxSCAn14QWmLCaxzzC/ZxeLJx/0PH3cJw47dlgfFeu8abbvTNqyF4thcTIlwoeVh1grSR+BZCEH6y3vecWardBX7jmMEiyUtuBtipi4zQtZSvbmeRxPt4zy6xoyvtSfA4kb7GFUBpFE8val+tsPQcBNs/poX9mwVTgkV4ljMrgJPJAwvm7NwQ/faB4ulaw1kO8cOkTsk2bDW4vvKDVMnWibam1k099CJljKfB6aM7aYmbl/76RF+9WPLcUJ3ARoSjcNfjD3F1eAvf3q7jMCqoA1E15xE7yOJzBSrw4ykDgHjr3Wu/YtgZZxrG5DkQYL9iG2LwwVnDJiEQdxH8e1sHt2DnyC4pwGtoddU64mJzR+bBMm9iwLIfTe5zQ4lOhZIJpCMhCROWAYAO8hsJVETkrnuvPXxD+FRbIDUt7+Ff3w5PLo4PHY+7l0cgCLf3tRvHjAb0y2NH3czIcJsR3JC0XldAGKlaednHRxsGufk7xjmIl6xWPUfpv4yNQ74B8kw6uFjj/yCKX13d40/jg7a7U4OgEYbG8dR6K/D/ODOk/wxVtaGsKd9Y317cykqwpWOLYlG39s1fk2+vWli0FzC0d/p3E2AyZeQjlC09j0rx7GRE2CJRnovz58ZzjxIUmcexa4wnNXYzRrjE9C81Cl+pSiJ0yuOIJbASjq3EmM6+gPMhUAD5QKLDMCSetPOtukfK0pou9zY/wTTWgS8icV9+5Ymi5+2phIAWFMHVvMHh8oKsufOH9PKlcfOJ2e7yyUOhnzk2J/TT2vZ5Mk18GA5d6c+JvGN/jAmWxi9kYGLfUiwECoQUzzxU4sBNxoG5ncJ3gyx8q1ZtAo9cOTVOu3B/rL1mL0tkd5JCr0DGogXTNNL7OCmkkoXPP0rtT4iUs8Egx98FDqB97BrXLLm0jRElblSKochxHuyC1Gc7wld3VLcRo8glvwwawhFdXcTRuCSH09ZQ9Z38u48pSaGgZQiaTH+9maZgoS0cWZrLHHLDGXf+o/Mh5VF+4XEDHQ3yCy0oA6LXdEBEvvYZpWpKslsmfz68scp8fatnUBupVhgkKVRbRXMDGvOU0DrhvG3sTHgI5k/X7YxbZI/Sud5s4kjix5d1YMwfU37nmpfnVR1M2V/dFlX/QPOwHyxdDKby85Vo66pkm9f3fnKP4jjKN4IkE6SIFz5de/BK5cYkPhixiJBmMu6rW4cNKRzS+0GOU55qkJUS37wZiAxRaxXy/7SDgoldCr96MuPPTeOA9LBWA3i82TuPvpxthl0nC6L+QTMlEET8Jkz/XC1wKGEjaoQ05rODEwUKxtb2U90auar3eRDoNGsXU710sAhS7oeHxcM+aoiLxburZxUjrINLPrQZEWb1Owmz83huTU1ZY1U5hCW0S3W4TnRKVLEn/sLEh90LFwXrHbDxttYbBP+oL9xvGChv6TP2XsWSHgpEN4a+vusDc0vm9QDCeCl70D5/p6MiRxN0lbChLY8jpaIz7Vk/OOpvBEIuwebbSBsT/3NL8aHIE5SNoZd6HV3vthBKSk2137oM5nCXYPNQUira9LnIGi+nu19Nub+DDol9JEySGW6LuMAXDJ1w6k/xwltQPfGiSBKDOsWLdEt5iCuaLdQ6kMzNU5OPts6RPAQLb8ivO0oREkgPNiefwdlzCYDSKTCdOW5396s7U8RzPTS9hfL9NGZutgQVmm31wkKNbhST2q3PfRFbK+SkvHtTcXuqhejD3ZI7PEfY6NLJ2Ni/K4x5OvHmvikaxY9Nw3GZiKNJsSYQEV1086Qhi4T8j+eIeRGLMCUqwAb18jh7n95v1ePS7EdaXMTJ79+HPhYalwGACMugzcNffsUXPn99g42HhnbW7SJmjiLFz4dt0k0F05czleLibKf9L7fmwYkyF22iwHIgdvln/AtqGNSKRZYynGrszZd6ngHr4J/byVN2evNfKvXIgsCFftcl361RPoO+N8Ffgezqh+i2umWT/mf7PT9Hr3qTTx7CvsK7GcJiRyis1IyQBhmrXHGY/UIvRb27c/Dhj8CwAuSKlHYgiLx1ujW9agYofb01o4XicMgeM2dG3c+c6xNZ4awTDFiE7TDo/uNhwwsx2AehWOdfHhGMFJajzF+1nZubGSJiqgvrXzNOx9W10b+0tCyvm/k5Y2cpi2cdft89/WNS5sd2vt5oblD7aGPvO3yT5/IlQebXPlfzaQKZxE8wAwmv1rA7kjxbOY60P+mxQ7+5XuivBsLqCvk6TIschdVge3DzXUmhh4daMq7QA6jxjs5QgMRTeypJGWfzr4Eb0jdCr0qvDjz8LA/zFX+3iCEYw2eANj+M9/3yEvAvqsEAhybAxIHs4E3oQoNHoQah4095xmDRgeZm2v/9Ium9vHgVVhRjHFduHlzZ7Dy9fI8yEEtR3oKlXJBfl0xq9hyXJf7mS6oqWM4aWjJHN7lSqiT7sd+gMzq1v8oX28BA5K4jpfXO94718tVqauSIzCQrIokaULqKs8t6C728kZcPijakoomo9S0hVRTL8vvtPlsPiZqNgZWmdR2zAP2QsmjvXaC8RGxCpPvK9//l2+1G/bSjxfwnFvtptFtGjgDeg2bbZfrVbRKrLWtLAfGvwVCDsJ/CwQv3F1GrJ8GtU7DpctxhMVqngZLBFSw04b9dbPYw5BWYQUYLehqv2pmQaP0qdo/CwAQkARKCOA1jK2tZ3pgq6DRRnH8AY4Zz8IhQ7OV/wnmRvkr7W9f1f4ub3/3qvb5kkgn+Vjbc10l51gJYgykg5ANTu5qe+oHcysnme5bb6xjcDdJjEyfna3mcyvv2Ww0dbvtBceYkSkS3/Ptz9ryWEADuV638ROrBLwuWx2y9Etcg+Z1lKD9O2avqKWUzefwvXPjgGwKmaiT4NBKxhjwuT5ExDH987KZS/DGVr9p0P/PehQ3hsKcu0b9FPHvixu1Rjd5htA53Mvd5KqOtpnZiJ8kWJkG+Y5cQ1M5O1hhAVBR9AlrFUMkEBIBXWNp5SPSrRUyL6oHkdg0KmzMviRr9u3Np70/yXGf21DKxoV83zV++E/kGWSz2l8br+Rzr3G5VwTeniER8S2JiGMytIrYXZMAau2jgRjjau6xxUFeFdjxKj75qbemmOX+AXbCQ69A8AV6jxHB5JeYqnKGifMTuhtmnsc57Z+gC51zCuHM31+hSgFMLWHm+6l4TTSdfLNAqHOq1DwrsYDS9WxHUh2cbHxy9qlfK14+1aMmXvEpoDjWru4DchBeBSc78KbPZxX3Dp7kXpFK/KR4T8Fh+ND99BKqGgkmcss3qq4m3iVozzZhZa/s1jhl9LHX9idtuyCcRXrkZm//4vDrASd5iLNi/KtEzfFzC7+S+BhH5BncKHhe3sUVimRRKn2iRA41UyiuiR4OykxwfsyzhJavEbF+NHplu55nKXCvIYbsiIw9shiWBqDMF596JA8ywP+LH8JMuYk8fWaHx6dfLpyLP08Pzq3pvLJsivV+5O4aDy4efKwg++0NG+H0Dvh+Pnl/cATM1g+Db29Y3HOgnNquR82a68ClYxpFpHNRt29vSokttKNfzmHZHCnIhyjSXGzjlCJWtJDKPtziCSZbHIaDv/upbnKF6qEo8eHoZO+C1mjDpEB/F6oktaY0ADJu4JaTj+0Bf3Qf6CMyB+gT5U3kb19Bp3OVOmBLlsEzSQavII9FJM2SEcZte6exiQ66a5HpACv0dVTot3eGGhW6g6FGhOFg0BvqZMDryvhPT/rGPTu4+HJ2LLxNyrTGmlmTD1+OIWFOjjkGTnb0YxbH2MdB8vHk7E9+A2N9bwrHtDrAOCye7yDupoSDSlrK2XxT+EA3VD3y+HNHjvKIPyuaUH4VUM4Tohy1xzRTMbcTMkPZWzS1WJoKiJTElLewHYdFgePYWdqDeHBV/tyn3mcZiBy9aoTc67omo2qi7SQCXaMM7Z0SJcRr8DDcuOE1yZJGrTplCRUb+jH9SymDjzwtknPWbxE36NEqTmFIKFBAuURsLec5GDq809if+eQLpmhX5h2m8ER9Zk8WjMiTNbzIT3R4NG1vRQkc1IWazt342tfyLkieIBCyRdEQ3a2Sj1GCF/sLyjEptEPxXMPozDKV9PwkTk1qcWoSTurVUptqltNRmJWbY4RFryKe/kLAOqw66plaCK7oGQ+7axo6hYQrmDWID+l5rkFscPGX1L56saXxY4EUHWJ+gmXOBuZ3UBAmMAuyjV/9A7WSsjWNH3UYP2390BB+yvaLliC0WOPyuqNdEk3LOTjluIUFbSdbpDeblNE6FLXRaxxdxGcIadz5Gpnca5dciQYfdiozpcaxwm7KsRZn1XMQfj6vZW1GS7LU1tNZcjxrfGbgq6yH/wDdIrGWdZCrU1jWN5lAtbqtD0Zxx3qpuNk6KXGkCBUSGpzYAq0qn9jTq1iGsl6IadbzgEUvzPFs1KydyKPxeu7Mpl51GOgjQxvN+JMGJ0YsoDYqDkk4xBG98I2JD1bxVS4xzcUNHyuiVzzNhaPemFLPhALFOjxY27EIXdDSj+ePBK2US0wu6CYJcXgT3DTLoAtCJATLvtPBIduQJD2ZtnPwgJ7xwJnMLvAClsrY3ZJk0glMvqnuxZa8RS8/3Rl1q7HWwGENQH34WR0gg7GmCxTAX9IJKiAUW9PxL3k7+tsXZHSdfC4YhgngCTWMG5dCAppHAdtBUpHFRc2z1BflkNnkZ/LNEbPPxuT8EQ2YcIQkT9ef9yADohVIwDc+0iGRnceIgbSWNIqng2M4gTf33zFblSM5HOuAbhFmE3RJQSIKkRagxHVV1YHKyYcTR3g4dULOkyA2vmexp3xTlCMtGgsqwlSl9KaEnMpaVxaXw97kory1y+TUs1OKvBY4Y8t5LFVJqpDbLEZz7EuR9CwII2HZTp37L9sDktfzQrLQxlD5fRb7kaxZALWCWqc2Rrx/FtKz2rLiuUzbJkFK/o77YD4vfEFAuH7sbGHWkvYqGbM/k3mG+UCFnPvWvRCZN71VSohsbJxnnQh4xrbUz0WZ/4/gSSnNNLbxo4zKE+Pyg5F5av4QbJ4aNYkjYlHMNrDPettfiqAup39kZyrSJclEKIdSpYTi/oYkBXZlHTQ5LlIyDHHoQJ+aIF0KoiDlUC3LCZGEWVkKA7VrJ+FIMs54LT0O74jBcWQjNzQpZciRnSD9asR+KYmpfh0xapA4nJ1D+RA5ElqGiSpiIZuA06Qs8uBIJCRv37AzKOB8h200q7ExZ9K6QBfmmmaJhxqvw1K8mXUMTSjrjZ+BV6W1nKBlStLpWM6Lqol1TQGkekipgECTz7GxRK05nlyhWo044hIoCdRjr8cIn23ehdfTTExSOYCD7YMxAF2yuupjc79ohxvnH6wmCVlMMKphJKetv0msS/m7VQKAQT43+2cTadUgYw1PD9Se3ljl6I6Nhs7CX0Txo9V4Zq7C/3AKxgkSxtUOf3OVB/uwCqdZC0sGb+bhE6alWopn0nnz0MkG1qG6AHEEhExECrrhbL1DhkcUU+ZFSFIZWX6zGadfr0PZJMevp7ompEJUpL2L59+quoQkfjiuxKb50CfJE5P3xdKUuPrMfgpFrOmetsoUh5C29fK6dQeiFtHYqNqKobF+fkM35J1XZORCxFUiJZvyt2siKWtRlmo/pdlsxPNlOtScP1rW4vj1GknNSjBOiBUobBx+VHuocplA3Yufy3l+MfpdRIlrMqNr85/XYK4FRH9RejaUrUdNEL0TNZ5XKFOrM42akqZ4MFPKYcvgsoqNyPiIvCVYgR40eJxXXA+IwJ4UX7EX0KcQZMzN3RlEhG2cu3d+Uob3nz17aKCQKqJgTwJPJOQiJzZ7q5gL9dwFvH9kxaYMeQqZSY3pnPzUfV8GlslQw/rP7Z7x+Q8UrfqZAki/kVezJFXrDPDMwMxNb8Mq1mT8q92dNViVvFaEVTlXkE9Cik1hM0006A7sRk1oKDMZxKX+pvkGI6VOLCe3XrpNK8p+NBLvMYJtyTvO8Y4oxoKk5yafbxSfzxwb5AyQvZbVF6iqmhVCmzSMKoHMPiITFMRTbRitvIfCR6okJDq2n6Fh5f0v8M++W2SRkXOfp5P2HYojJ444LBzyjmHfZMHVsHRC88i2Ehxq4KZhog9V3Tg4EAJU4VKmqtlUVHTuqFJk/MMsYuRiguE4M3cvzYs+Jpk7gs0rRD02DYPfTC4Lwi+Vc0pKiij6iPkjWnZGinpy5l/7D5auagFKZcBaT6xaYvY2VxU6vFSDIIszdSjDEcnYDT1qwoSlLQJqTucrFPo7Eqt3COwaaUmGZkNyL1SVATmrN2wjNDZQ2SHtdqMSyBYsDmAyz6340sSgqFLGnJxHP3Sw0ZxkhcBOkphXlL5S25kVA+qckj8fJ4OfmlcvjCOlmGCYlUddKgN1RuukWIW3YXQPnChSQVoqEY7cHGuk0MY7TOjMsOoI0yG6dCSrqvEa3F+ClZG48SwVG5XdsMZvVlEVteY8qhmalWvLDF2wNc9HcFgjH1S45Q06UZlGwqlRVbJdaxzErS1zj9J5sQs7z63m9aXZaZtXlyb7ChP6jaUSLeSlOTSb7auXunf+re5D6iMhO9LdqaPJsX3zJ9GmM+U5Rs877lT60cGGRaZgIY/uw8sU66OTasnFyFSLarM0K0pg9Ub8ilpR62KMTV2UOogMbJK4OKQtbtjkOrZNJqHO06F0QJIYFJx1rglCvA7gAPcj7Iz4cY1pcpwTQtr6Yd4HXnpj7g7b208IaJtckQkZ3ekO+PMN6xzmbvup8Srp+8xu0HeE4lfC2eRd8PatYLU+JSqPQmhvRSreq88nO3mE3maiTjelnCWMgYmGva7Z+O+fIYyIcbr/7OmhTDAcBFAB5IDAoU35SyoyjTii6TjkeXUcUwEqDnw8pk5QO2o0juMo9SdRdNtijSPiqjLlWaUgLml5x5GxXE0QsoW/K6R0MfJ5E96t032YfSh/ltQzpX+84saFMNc5YMPczINJ/vmfyNIp37Sg3bMgoEl9R5cM7qneKLkBLvPi42oCZxbJ3OLRY/E7xL1cR5F9Zu0n05/iOUaxVTS0eLwKPML/7OTkArKSRrfMLSIedJWtG/cREr+l6Eybf//k84c/Hfz7+ZDaQ9zM/Mmk3fG2J/52z+v0dzrT2cAf9Prbk4E7GU2G0+60veOb30Iqj/tydkS9btIUd2VsbXH8G8sHHTAkbwNMnii+3lrCNoHDLdkaTbb67tZsMpjN2kPP70y9Tnvk9/HJ7ff9HW/mD113e+a526Ou1/FHQ3c48d32oAtcOqiGGA23cCq0kDpj92yQx77+l0Lk/NMeIdLb6fZ3+p1pe9uf9br97sQdTLYH3UF/1Jtuj7zt2WDaBvrT6XZvZzh0O5PupO/ubHf6k85gOpjQtDi9hGD9W5eh6EX6AHpwvA+wZwT2+70f9u7mTq87cbQhwrvZsu+4998rPb/uHVSQUYi4/nMD7n15f3hS6eiuvEA6zpa9bqXDL8YZDmukBcFLA432jgtmKNhGqcZsboHESNaR6KTkaMJiI5hpHg5C1oRLNg7GPz8/uKCEHaohtsy/B/90gy2VTkK3PlH5ehyQAu3OSVLN+ijO3NnZno1GbX+nN50Nd/ptdzbEqo+8wQir2ZnNtmfbaMsLhC4VYz7ZMn9Tr7rd0c5osNPf6ULfMbe73Y7rDwfe9mh71O4N+t6g4w/a/ZHX3dnujrydWWc2mk12Ju5s0J50PK/nzUaD2aDT7vQ7PSjOTZ4Ch2laJ2DnYhqEeX867XijzvZk2Bn1Rzs7k36nu9PbGfR73Z3ZYHs4xb4BJ0+obYoQFYyHKfCNCV/FFs+96QyGI/BndzDAiWH2Bu4I9Oi0+/0uKOL3et0ebVZ32HXdUcfvzbrTCT5gE/uTfn/od6d46Lf92fYQr4bZZF5ej53poNvBrpgMR6PezO1Me4PebDScjdpD19se9jod1x24fbQFh6ID2Apo8++9znZnuN3e6UDNM3cmk663g8E7o+n2cOJOvNHAbw87ncloMHVB916v7e60B/3htN1x2+32wB+5A/y/M9jp9aH9/A+SX3DkrVH9NGyzlGiP8GIE7hh4OwPPA/E7Xn8bSPd7Hig+mnn9tocpgaF2/E5/2/cguUDxbRIuQ9fve5P+iFG+Kqr5K+ejOgBxQELY29lxTRERE56H1YNpsCKWVoS3yT54lH9Z7Lc2r6Poek56NoCaecoqIKoAom53rUdzTHySElpVuv2RwYkZT9D41EW+Ld09Y0QS6ci0AdtsFLOrsYnzOa5pBjxxZdwnchZqj+RqIn4iRVtaBTfmnldWZ5CW7AAqNSP7SDzcFCcCT6co9EOtEAcUAZOyZ7Q4qhgoAEHZUZYcXGMiImZNokVV+zZY9+RhVHWI46gWjtM0aGT1POtHj2oTfIDEmPG2STniyD3PIkesttfUXbqSrT8mrdmqn3HRCjNen3It5Dsov851MEGWPLIuLEEthSo0Vy77re7btz2yGxsVZNu18CrgeJFR4hanqxyi1ZABCsi1kPLldl4HM2//AlwvSG7ZKNdgimJm86tVAl2ltO+ocQFUMV/m26Ez7irf0rQPieiUPcW8BasvXxXYYbgq5XdjtFvx9VN2jLqBx1TaLeWq5KUhoctp7X98UBX2hnX+edT+DXWnF/2tCzhGKYbDufl5ouHkUbY3X6toZ3Ye57gLWoqoJiHUaz+HEV95Fc1mfEMVGCBPw5OylZTSa+Al7bWNj8EfHGWiDMl3iFzwAyqs5QyhhG64g7s0lhsRA85SccmXCCvkTqa4jKDIPzZV/ukCgKGle3Ix3WpRN487fSLdZydy6gYeG1jPzKIrs/jKM4C9SqZBiuNh9NAbkNui25cbfJIclwyROr9fXg2GD6VCEOlUSFHOfXUKEJoIVRKqVs6uUaMWiReOgm/fQlMl5z0HpdF4dgYwKqy3cFBBZ5zee3xVSTaHws6xqRVVo6B+l+3yBzq1qNeVdMPfphAjK7/N4CsfZenOm0Px4VJaDpzSkWQBzZFZO8mPsq/I0kWWMHGYn8gzuvWT5Ccl/HB10SKYP1buv1FEX18WlZUO88pe3NJpzDmKTnSrlwtzxQK6U7MtMytPLwkJKWpQCY4lJykoZF4H5CMwOe+Jfmm1wqjFyGA704PM4kLDm9UEWsBiS3Q6NZgtEHiUxgbgGjiUTXDkFr/rFqFaEw0InO0QulRKVKypEAmQ4AeyLjPwsCFafOcLffh0sPfevNLA4TSFxis0s8EPwVJjZjUINCIdmRe4+IBISSGHLJ8qM2I4hRAc/I4L11261ya5UdyReRyyvcwZ5XcUFFoGS0MuPcvSDJFzCKw4+JHnugkfbclPNdwWG0WRkV/hClHMo9naPWOi8ZAFn186kF3FJxpExkjgcKrv4Du6hJ20Z3b6kJprLe17lPv5DhFY39wz88fyaTz+oQZAVu2TWdw+B8cUzYGtlGIy+Ueektlo/MYAM0kAbiJlSZwsdPQSK7HnCeQzJWBCG5RZbMr8mGNJj+KMRbdMJYlJrMm0iMlXfMfjDGU8Dric4T6y/yzjRmAm9V5CGrrLNXcLsJep5CWx8RHSC3c/3PmWcl80MyAF+ymnD+UqweC11Hub7s9yJjS+1WggIe7BC+DTgBAmRlU+iBd4lA71DGfeNclqQWVpXNRT8OA+X/5IIi32r1eoQDBOWTPNHClSZ5Ap50J9X5UMLB8h3j5FS45EroK5Z2t30SmPks3X0WXzpkATXlQcpgtKQY85x4kI6qfyYN29m10gx9aKNGLVFmkf9Ka2ZkBdIwfFg7SD7Bo2NoGwhLkBpKAFCSFd61fOE+DKVeYcJ6FqIRHEW4LcWzXQLo4yW2XMV4NeQFAkc/4+M60EKE2KWPR1+eImCoPhZTH0dSf/oNmoQ1jBVueKGohFZNOoO2XKOUVYJWx4ioYytYR2vLhJPG1mQ/DKm/cTiS+iVq5mIkoZppwjAgUvs8Ug0FqNrLVIKbrF5HU47x33r6p5YDvqF4EXW32KPLpo4fDF4MolqRpg3olTN9WXJQ1vp+cF05QlkmZq0lVqVBjNVya/o9vLwsy0Ktl30kp/1jDXj0ZLZJCmn9nkLob9oMljemJ7q8UyUcoWpZJRFsK4+4J+5SGwyIn+4u3Kt8MmFYWPmYoEzc6Ymxl9cjKQ0kHgVrUWVTRp6m1ezaHZ7dSkLiwsciWybAFJ5X8wQCouRnykTabchkpHspRjj5o1cl2pENTrVpt0EIuNb1aT8X8zRm/ZfntBPkOfXM1muL2XNAACm9drwYSIowcOmEIs93bsAZsJ4vaE5KWbxBJjxA+VtpkrFTNlDDZzlYR+W0R0V7dOATlPCjJoyOYSTaNI9e1/R3z8YnxJlF+HXEuUZ8M3FKt0+tilPBZkmRA9Pn0w+AI9vptYy3Dlh152AlRYy8omLtPNCDAuKJEPPha+y3omZqNc4SDj2JTHbZHOyLdW0JlBP19x44J5Tu2z0xa1OoLUky6MKV+ucvxXg+EKDRGksRKks5owqeQBYTUDylecu4uJhysaZqxHWKO3nXa3z//QpSVmbYDtxl4tvfyOzTI1bqoaiPz+GjrsVzWPelpQArkwFnxvyeMCzsnbtUO4aFQcmEBGESl/9sryKpyX+b22fqbPyx6j0YGnDFbCEwafiAuFSI1NpW0SNRG6W06QfL60rhbL3NRYUcV6MAuyAEg9gnKumF9V012z2A8zSI2bqqUrgWLIG2RjQzb4lObKIevxkPzTEp4esztCXBFjeCU0tzLRQnpnlhUpVAzB+H1stOmTAMk/ystfjV5Xe5l9lEEgTgfap1+Nzjat9GBNpBbRcOg/bHFlU+l1kQ/4gMX0KNKQyN1niCx1tt+Gvw0yyBb5WJDvP6E8soGNYVQiTkNzuagrDdTQZscE6fX8il3zy/HBxRF97wSMjCxHYfeHiWrbVJIgdiVaZEqKoWRso5+qZDefnnLvodldB79/dHhaA346D5YKfBbCMVUPFYlDc3HXbhirtz4Woio1Q1EgT0bCex1C/+chcBREhzFYh5EXg58H1wv3/CaYpSVokhGDTLcO5cM0zYRaSLSScitw+4F6JLbkbs9u6yMO10eUBJnaMQYyRl5uidb1t9+je5H8iJwOu9M0izxXc5duK9DR2H5m4ocL6BgXkRSwV1edsOoKVqArferJJ+F+c1ftgiz7hH8082QT+dk0YR5d02vZDDpmo3XMzrB7osVxFJCXRcMnpCcOSQ5zl/7Vweysg/n7uSQHnSPdd1pe1FLa0K5KGtLBIUK3Bu8PNwmm5yRcuXntAg4Vk+Stds0koLHQXHKcdvu0U3gq4J8ys3Q6G0b9iPyHF4bMv2MFEpweb9PjEvCa7a4otM+my553R3lM3jrNCd5IsaZgssupYzJXgUGPdtQT2kmcHQoy8hMqScUFpAFxmkKuU0auXj689yl0XLN0Ar2GMUtA+88A3ePt+jrI/TXINYJkn+un1ncRT1qgqq2jJMUlIcgPZsQXXTDGJKCo15J2yoiWFCXodK8JS9Xk7OMfZR4d1i3oXQ0KSlpdEt4yIGQLZzFR6CJ4QEcxWBE5plQbm/7pW+wOQmOqJE5J1C+Br0nEm1KP7rBf4POkOf21HEl/7i5Rqeqosw7nvmRj4oa7ppFlaE5XFGzkX5FlnWebqeQ2+DvWoGzqWfhaw0dLbv5CvT15WpCX5ZN3np3zv0M9yN30KluNr63OkaMmz6oA+5gkXd8ULKTSTOsqLmOQYUXfzaKq61B5E/rXbn7ft2CUH/2cWkvWRjEz42/Aku8eKyZIz6QDhxrRvkKaLUo6flsAeat35kAnj1OJF3MZw1hgbmkzITWYx6nW1Wlt1kLPrMNw4Bnpd+xXULiNn2EG5WdPxhp/1FQ75cm9GWjMdMwoNo2Nmb9jml4NtCLvmFuQ1tihKfMHfcbPhZo3Z1OPaSm2rPq1aNQtRkMvAjiXGZYCQydAZknOmiTlCo/PLcldkJxBushmCRnEunXCCfzqCiVly+pHhHKAlOJDdbezlZNxVWxkrWibX0rhV80LjVvGxL/6/USBup1IvkVk6T6SkVwKs3FbiiSMDc3NpFrCf8LZRZYQVB7WL55eH0JXj1ac6Gfy0+IYFEJQne62TRl5nd1RZ7Rj/iYIEiJj+kdutMVaoDRiX0LqrQuRwXznsXwRwBa7yp6qxva6/57tbnwWiYLNMO61pVbCT5YUSd6tvY6ICaIcEtKulHXLgd71eoWXwoCZH0pf8TUJoL20Ed6eW40N1fKCai2vKN+iSnnFM3gVxQPh6pYmBVIpH2HN90lJthQa4FDfnHcF/Z4vnbyg2dJvvI4SFQwSgtByV2nUmrt065NZs7fR0udSyZb3iNMymLYoKi8g1AvxobaoTA1eYXnlJo/htKUyC+hRV57zLmzN45U8u6rc0Scx6rX0EE4A6I+q9x2AJHkCQDEhqUtrSfKIWfKIIZ94MnfDW/jbWi3K/MS91KcjNn3hGoZ0uIGm3uKiseKLLLJrUEQ6tygJn67W5Mx0t3L1ZUVAaJHUUxWYyYLo5fgxjoHUI47PGIMfIJ9irIE4v3h/8uVCm5CH3Uc3VdOxRaHORQT2hGSbWo3fOqO2tuGkDr7SBjTNIKy7fjbwdvHdBa/x+tTxNsfa2GHeuGzhdGjvXlVDJrU171Rcx/dcUnI7RKa5pcot6YXZqCvLlysR6RqssFyhIQURXA1Rztxv5JUOT7WVDjnMFuOTf4fT5kIHxPkTv66gfqML6rPybHMQhf13dWM+1U+5LGb0wv2Tc5X5/3xJg5QbzPHld1a3WoayNhfB/0KktVIK/8GLDeVHrp2d0DdT+vydqI/vskwXo+AIlXCYSWsliP3sRPw5T1lxcoD92/r0akR5donOz3F6XWzhnHOJ5Pv8WGGBZovv4bH1JVLnHJUab/IBZrPKZqSTn0ZgVq5u4qIN4K+4ibZFlIOCSl8yX4VChFNQAjZI4UmoGjxP5VNQQPMNKyaxpePTxOHGz8K5egMZiMC+4vYhyheSznpIYBnQPOT5pQbwqvLtPuVYaa2Qa2Wk+z3jjfoapp+Xd5u4QW0A+KqpSALXIlVZHkW5EdiEkkzoukF8QRJfg4xsuIZdt6t9+n7H8roiw46y+WGqglQNpjv9UjszP9x8iQhFXFZ0UnEjWT95RjzztEFsBTPVUe+AW3hQc8YFQcwd5u7mIvUNjCCAGq8blVJ15j6Xv4oy9hxGWf3jMzjJ8e7wxXi5VpZFSuWluTG3YO1bF7XU0dpbdbBvyAfOtzvllFf2n5D+uZq7Kij6okBO9GEnBwAgU2tGP5Tf5SUY+fUTKe6lCPgb6xgjwEUV5NVrumu3UlgaKbcIpFqU1UQuFwReZoNfXOZOGOQgvoa21csEtTUD2nyxoIqOqm8YxAOO+z4wG9jkvSne0F0zm9JPNv35xRDXWXaHpK99t4xSFNnpwRXy6x4R+/UjZdVrd6/vwoaVeyf2QyKXzN002IbCxQKB+xPz1AJO3NVWlzrIF2qz3CcdTt55TBKLX43bjbXLu35q1I0ZVjxwQLeD4iqjLbpIEKOSym82Xj9C6TtRCxJtBpBF3ajfCzOpwxomRx5I/HzaNzJJshlS1V3w2zi75+8meaYbdg/iTVm9Q+Zu2nQciuIx1vWqkg4yLqsiY/nRlPS08Uu4MB8qEVqY1gknzcx1+7YuwL4mmCTJppxfQ3MtZyzW00Tl21DzV+u6mzTbj8U3JFBPj9bynTrX6eYx8JRXpFngm9wj9jvJ7WGlk13p5Ahr+JtuLdpsPDBqus4sraoejv8550WpJRIeEGMjDf1VVlup772LgptcQx80ag0VzeDNyP+wJPvnJei3Ac3hNSjUuHhILdus7mVNigVBSXMpBW7r6HD/4Pj8ANXNVG57BKdmmORfU0BZn3J9jTVtoA6hO8zenBI5xWbA8HSvKVKKYSGEvEc51ymiEgJy0zaZuXDrCrk/0SGaZL5NlxN3CB5/rzh9jXk0S+/5y8foRp0kiaYBX2znRdMVlwLKF33QbsalNLBccBu96kJfPE0jwTcw5+9rlwqx7HV+Dwx2FV3bnXKqj1wSQrjk18QEi0ANw/f4yDeB0ZeVR5St3WSU+Ub2gHLGccMlzZDLvRNyMgYEHRc/03c7cg04EbRJ89kSo42RAxD6ujf1deoZjk118R2nOOErTIRgVBtA3567KM8nYKxmdPt5ciM72EOKd8Tj8pWmaaRuDYVP6F72tHIiy9dH0N8LyiSbRHeSQyWLDW4K1Hen8KIsi8VWr5CIhFskJ76in9w86JanFvNFb/RF8wFdKg6dgLMrKlO2c0Q+HRjnJx8u/rF3dmAcnhunZydfD98fvMcK753jAd1k9Y/Di09wJRloc7Z3fPGncfLB2Dv+0/j74fH7pnHwf07PDs7PjZMzgnf4+fTo8ACPD4/3j5BLcPwRN4dfGMcoQj86BK8D8sWJQaMqaIe4zR/wPh+c7X/Cx70/Do8OL/7km4w+HF4cE+QPJ2f46prTvbOLw/0vR3tnxumXs1M4KIDEe0A+Pjz+cIaBDj4fHF/AgjrGM+PgKz5QpfLREY1G4Pa+YBpnhCiqCU7/PDv8+OnC+HRyhNSMc+OPA+C3h6/NkdEwu/2jvcPP+EK1vc97Hw+41wkA8SSppaBp/OPTAT2lUfeO+QtO8PUDmM/+yfHFGT42Md2zi7z3Pw7PURe7d3Z4TpT5cHbymWdK1EWnE4aDrscHAogob5SWCE3o85fzgxym8f5g7wjgzqmzmmvWPl/nVquV/Uo3KewaxIF8wXx2MSbxXen2aWZCuQckofTE4DoUfufqidiXb2COEZ7zJMdUv9NfFa8VN/knfL8m3f0V+9p29eSKZ751E/BUhgW+7YSqNxZ0N8RjJh5xjZcvF7JmNS9aUitSkLfy8YvfWoJHS75YoEU1HfiSvdZdhy/X30I63WSLXOKZQC72J1XMFXXwSn1X9UWs0SNYqjLSFMHEsEQ/sHlLSSDPVhdplcV/FjKSCy6YeDY78DI1nhNe8gVD9pADdjw/dz7vnZ7yUo/pa1ykFb47g395ytu+Pzw/Pdr70zne+3ywqUv+DRGGqjG0ii8S+Y0ZAbewMkjHgYihyztQwv+mBhcSEepF7cC4uZGJ8PT/AYgG/B8RigAA'
raw = gzip.decompress(base64.b64decode(PAYLOAD))
assert hashlib.sha256(raw).hexdigest() == '83acfc4f61104ab9010f53171a3ae247e7ce32cf5d8f3c5552f91f3a5036c065', 'Bundle checksum mismatch'
for relative, source in json.loads(raw).items():
    target = (REPO/'colab'/relative).resolve()
    assert (REPO/'colab').resolve() in target.parents
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding='utf-8')
sys.path.insert(0, str(REPO/'colab'))
import runtime
print(runtime.inspect_environment())

In [ ]:
# 2. 無料枠でも実行できるテスト（モデル不要）
subprocess.run([sys.executable,'-m','unittest','-v','test_runtime'],cwd=REPO/'colab',check=True)
import ast
for source in (REPO/'colab').rglob('*.py'):
    ast.parse(source.read_text(encoding='utf-8-sig'))
print('Python syntax OK')

## 有料GPU用（未実測）
SM80以降、VRAM 20 GiB以上、RAM 30 GiB以上が事前条件です。64 GiB以上のRAMを推奨。
T4/TPUではこのVSA経路を停止します。条件は動作保証ではありません。
[MiniMaxライセンス](https://huggingface.co/MiniMaxAI/MiniMax-H3/blob/main/LICENSE)を確認してください。

In [ ]:
# 3. 対応環境の依存導入
ENABLE_GENERATION = False # @param {type:"boolean"}
if ENABLE_GENERATION:
    runtime.setup()
else:
    print('診断モード: インストール・モデル取得・生成をスキップ')

In [ ]:
# 4. 約39.5 GiBのモデルを取得・SHA256検証
DOWNLOAD_MODELS = False # @param {type:"boolean"}
if ENABLE_GENERATION and DOWNLOAD_MODELS:
    runtime.download_models()
else:
    print('モデルは未取得。ライセンス確認後に両方のフラグを有効化してください。')

In [ ]:
# 5. VM内ワーカーを一度起動。同じモデルを繰り返し利用
if ENABLE_GENERATION:
    if 'session' not in globals():
        session = runtime.Session()
    session.start()

In [ ]:
# 6. 4ステップ生成。このセルをseedを変えて再実行するとウォーム計測できます。
PROMPT = 'A cinematic shot of a red sports car driving along a coastal road, smooth tracking camera, natural daylight. Audio: engine sound and ocean waves.' # @param {type:"string"}
SEED = 20260908 # @param {type:"integer"}
if ENABLE_GENERATION:
    result = session.generate(PROMPT, SEED, width=608, height=352, frames=124)
    print(result)
    from IPython.display import Video, display
    for path in result['files']:
        display(Video(path, embed=True))

In [ ]:
# 7. 1採用動画あたりの費用。実際の購入額/CU・ランタイム表示CU/時を入力。
YEN_PER_CU = 0.0 # @param {type:"number"}
CU_PER_HOUR = 0.0 # @param {type:"number"}
ACCEPTED_VIDEOS = 0 # @param {type:"integer"}
BEFORE_FIRST_CELL_SECONDS = 0.0 # @param {type:"number"}
report = runtime.cost_report(time.monotonic()-SESSION_STARTED+BEFORE_FIRST_CELL_SECONDS, ACCEPTED_VIDEOS, YEN_PER_CU, CU_PER_HOUR)
print(report)
print('料金0入力は不明扱い。初回DL・失敗・待機も含む推定。正確な請求は実CU消費差分で確認。')

In [ ]:
# 8. 出力をダウンロード（VM終了前）
if ENABLE_GENERATION and 'result' in globals():
    from google.colab import files
    for path in result['files']:
        files.download(path)

In [ ]:
# 9. 作業終了時だけ実行。再生成はセル5から。
if 'session' in globals():
    session.close()
print('出力保存後にColabのランタイムを接続解除してください。ワーカー停止だけでは接続課金は止まりません。')